## Project Phase 2

Team name: A5

Team members:
- **Trang Kieu:** 
- **Terresa Tran:** 
- **Wynne Tseng:** Data visualization
- **Mei Wu:** 
- **Vivien Wang:** 

In [2]:
!pip3 install scikit-learn-extra
!pip3 install networkx


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip

[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [4]:
# Import Libraries
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [35]:
#Impport ran 5K JSONL file
accounts_df = pd.read_json("test_sp_accounts_w5k.jsonl", lines=True)
accounts_df

accounts_df.head()

,SP URI,SP DID,SP Creator DID,SP Creator Handle,SP Description,Account DID,Account Handler
0,at://did:plc:yoear2iyg7q4cqsojj4eej4z/app.bsky...,bafyreih4cp7nlqetgvy3svyk2q7hx7snh2q6jdy2sxe5z...,did:plc:yoear2iyg7q4cqsojj4eej4z,epsben.bsky.social,NaN,did:plc:upkgvxvhn4ebia3arqy4ru65,jaketapper.bsky.social
1,at://did:plc:yoear2iyg7q4cqsojj4eej4z/app.bsky...,bafyreih4cp7nlqetgvy3svyk2q7hx7snh2q6jdy2sxe5z...,did:plc:yoear2iyg7q4cqsojj4eej4z,epsben.bsky.social,NaN,did:plc:jutmjylg2h4tpev7wrvy2ay3,alyankovic.bsky.social
2,at://did:plc:yoear2iyg7q4cqsojj4eej4z/app.bsky...,bafyreih4cp7nlqetgvy3svyk2q7hx7snh2q6jdy2sxe5z...,did:plc:yoear2iyg7q4cqsojj4eej4z,epsben.bsky.social,NaN,did:plc:wo3lxbcfvdptzxyvq3qt2rgj,taylorlorenz.bsky.social
3,at://did:plc:yoear2iyg7q4cqsojj4eej4z/app.bsky...,bafyreih4cp7nlqetgvy3svyk2q7hx7snh2q6jdy2sxe5z...,did:plc:yoear2iyg7q4cqsojj4eej4z,epsben.bsky.social,NaN,did:plc:qzuw62ra5pgoy7ohoaj4gd7v,darthbluesky.bsky.social
4,at://did:plc:yoear2iyg7q4cqsojj4eej4z/app.bsky...,bafyreih4cp7nlqetgvy3svyk2q7hx7snh2q6jdy2sxe5z...,did:plc:yoear2iyg7q4cqsojj4eej4z,epsben.bsky.social,NaN,did:plc:bwvd55e4hsxpuxa5ikrfgqgo,pattonoswalt.bsky.social


### Jaccard Similarity Computation

Jaccard similarity measures the similarity between two sets by comparing the number of shared elements to the total number of unique elements across both sets.

**Formula:**

$$
J(A, B) = \frac{|A \cap B|}{|A \cup B|}
$$

Where:

- $A$ and $B$ represent two sets (e.g., two starter packs)
- $|A \cap B|$ is the the number of accounts shared by both packs
- $|A \cup B|$ is the total number of unique accounts across both packs

The Jaccard similarity ranges from 0 to 1:

- $J(A,B) = 0$ means no shared accounts
- $J(A,B) = 1$ means identical account sets

In [11]:
def compute_jaccard_similarity(df, sp_col="SP URI", account_col="Account DID"):
    """
    Compute pairwise Jaccard similarity between starter packs.

    Parameters
    ----------
    df : pandas DataFrame
        DataFrame containing starter pack and account relationships
    sp_col : str
        Column name for starter pack identifier
    account_col : str
        Column name for account identifier

    Returns
    -------
    pandas DataFrame
        Pairwise similarity with columns:
        SP1, SP2, Jaccard
    """

    import pandas as pd
    from itertools import combinations

    # Step 1: remove duplicates
    df_clean = df.drop_duplicates(subset=[sp_col, account_col])

    # Step 2: create sets
    sp_groups = df_clean.groupby(sp_col)[account_col].apply(set)

    # Step 3: compute similarity
    results = []

    for sp1, sp2 in combinations(sp_groups.index, 2):

        intersection = len(sp_groups[sp1] & sp_groups[sp2])
        union = len(sp_groups[sp1] | sp_groups[sp2])

        similarity = intersection / union if union != 0 else 0

        results.append({
            "SP1": sp1,
            "SP2": sp2,
            "Jaccard": similarity
        })

    return pd.DataFrame(results)

Computing with 5k starterpack info.

In [12]:
tmp = compute_jaccard_similarity(accounts_df)
tmp

,SP1,SP2,Jaccard
0,at://did:plc:22imtl2cambcra64j4k5bd22/app.bsky...,at://did:plc:237pyt6efy6mvz4rdfsdnmht/app.bsky...,0.000000
1,at://did:plc:22imtl2cambcra64j4k5bd22/app.bsky...,at://did:plc:237v5xewomz3ckenyp6chhhg/app.bsky...,0.064516
2,at://did:plc:22imtl2cambcra64j4k5bd22/app.bsky...,at://did:plc:23ngj6gipcnbf4meolvfjoto/app.bsky...,0.038462
3,at://did:plc:22imtl2cambcra64j4k5bd22/app.bsky...,at://did:plc:23sz5iq4zvn2ezbjt5ioeiej/app.bsky...,0.000000
4,at://did:plc:22imtl2cambcra64j4k5bd22/app.bsky...,at://did:plc:23sz5iq4zvn2ezbjt5ioeiej/app.bsky...,0.000000
...,...,...,...
12347960,at://did:plc:zxkhlfhfauvsbjtpfbfh54ay/app.bsky...,at://did:plc:zy4g6jisvuiddiv3sxp2mc5d/app.bsky...,0.017857
12347961,at://did:plc:zxkhlfhfauvsbjtpfbfh54ay/app.bsky...,at://did:plc:zy5f65xpf7xxsnmxqsljnnbs/app.bsky...,0.000000
12347962,at://did:plc:zy33hvaxxt5k6vj6gqzqlpvd/app.bsky...,at://did:plc:zy4g6jisvuiddiv3sxp2mc5d/app.bsky...,0.066667
12347963,at://did:plc:zy33hvaxxt5k6vj6gqzqlpvd/app.bsky...,at://did:plc:zy5f65xpf7xxsnmxqsljnnbs/app.bsky...,0.000000


### Similairty Clustering 

We initially began our structural analysis by computing pairwise Jaccard similarity between starter packs and applying distance-based clustering using K-Medoids. However, K-Medoids requires constructing a full pairwise distance matrix, which scales quadratically O(n)^2 with the number of starter packs. Given our dataset size, this approach was computationally expensive.

Additionally, silhouette scores indicated weak cluster separation, suggesting that the data did not form well-defined distance-based clusters. Because our data is inherently relational—defined by shared account overlap—we transitioned to a network-based approach.

Instead of distance-based clustering, we construct a similarity network and apply modularity-based community detection. This method operates directly on graph structure, allowing cluster structure to emerge organically without requiring a predefined number of clusters.

Below are the relevant code sections for computing similarity and constructing the clustering framework.

In [56]:
# if you want to run the KMedoids code below (that we are not using anymore), feel free to uncomment the import lines
# unsure if it works because half of us are doing on VSCode and half on Jupyter code. People on VSCode have newer versions
# of Python and Numpy so we can't run it or it'd break.

# from sklearn_extra.cluster import KMedoids
# from sklearn.datasets import make_blobs

# get the distance

tmp["distance"] = 1 - tmp["Jaccard"]
tmp.head()

,SP1,SP2,Jaccard,distance
0,at://did:plc:22imtl2cambcra64j4k5bd22/app.bsky...,at://did:plc:237pyt6efy6mvz4rdfsdnmht/app.bsky...,0.000000,1.000000
1,at://did:plc:22imtl2cambcra64j4k5bd22/app.bsky...,at://did:plc:237v5xewomz3ckenyp6chhhg/app.bsky...,0.064516,0.935484
2,at://did:plc:22imtl2cambcra64j4k5bd22/app.bsky...,at://did:plc:23ngj6gipcnbf4meolvfjoto/app.bsky...,0.038462,0.961538
3,at://did:plc:22imtl2cambcra64j4k5bd22/app.bsky...,at://did:plc:23sz5iq4zvn2ezbjt5ioeiej/app.bsky...,0.000000,1.000000
4,at://did:plc:22imtl2cambcra64j4k5bd22/app.bsky...,at://did:plc:23sz5iq4zvn2ezbjt5ioeiej/app.bsky...,0.000000,1.000000


In [14]:
# Get all unique starter packs
packs = pd.unique(
    tmp[["SP1", "SP2"]].values.ravel()
)

In [15]:
packs

array(['at://did:plc:22imtl2cambcra64j4k5bd22/app.bsky.graph.starterpack/3lbi7rkeagp27',
       'at://did:plc:237pyt6efy6mvz4rdfsdnmht/app.bsky.graph.starterpack/3lgyjx4gk362z',
       'at://did:plc:237v5xewomz3ckenyp6chhhg/app.bsky.graph.starterpack/3lb7awyj3my26',
       ...,
       'at://did:plc:zy33hvaxxt5k6vj6gqzqlpvd/app.bsky.graph.starterpack/3l6yllbjssr2e',
       'at://did:plc:zy4g6jisvuiddiv3sxp2mc5d/app.bsky.graph.starterpack/3lb56warofu2e',
       'at://did:plc:zy5f65xpf7xxsnmxqsljnnbs/app.bsky.graph.starterpack/3kzbgqgzsrc2f'],
      shape=(4970,), dtype=object)

In [16]:
distance_matrix = tmp.pivot(
    index="SP1",
    columns="SP2",
    values="distance"
)

In [17]:
distance_matrix

SP2,at://did:plc:237pyt6efy6mvz4rdfsdnmht/app.bsky.graph.starterpack/3lgyjx4gk362z,at://did:plc:237v5xewomz3ckenyp6chhhg/app.bsky.graph.starterpack/3lb7awyj3my26,at://did:plc:23ngj6gipcnbf4meolvfjoto/app.bsky.graph.starterpack/3lbst2hxkw725,at://did:plc:23sz5iq4zvn2ezbjt5ioeiej/app.bsky.graph.starterpack/3l73wh4ptap2s,at://did:plc:23sz5iq4zvn2ezbjt5ioeiej/app.bsky.graph.starterpack/3lammdlbdfc2z,at://did:plc:247eqkfslxb3ikrkg25tjrrx/app.bsky.graph.starterpack/3lb75a7q3au2d,at://did:plc:24lfqabkeag4lacxbqek5wgn/app.bsky.graph.starterpack/3lb5nk4tfef2w,at://did:plc:24t4ntsl7kggbni6h7cl37r7/app.bsky.graph.starterpack/3lb2ejorj3v2d,at://did:plc:25abffizj633cjtmbccp35tj/app.bsky.graph.starterpack/3laqxridxzj2y,at://did:plc:25abffizj633cjtmbccp35tj/app.bsky.graph.starterpack/3lbber662w72v,...,at://did:plc:zwl4ri7uarthxorlfeaomwmi/app.bsky.graph.starterpack/3lb3ojnq3b22c,at://did:plc:zwskr5vwlxb6kmu6pkn4vqrw/app.bsky.graph.starterpack/3lbrhu5oyii2t,at://did:plc:zwxrjqjt4hwuiw4uggudu2cz/app.bsky.graph.starterpack/3l36bpjvgwf25,at://did:plc:zxhivrlnevsv7lky4474femx/app.bsky.graph.starterpack/3l6oklpjkie2l,at://did:plc:zxkhlfhfauvsbjtpfbfh54ay/app.bsky.graph.starterpack/3l6thnofoce2j,at://did:plc:zxkhlfhfauvsbjtpfbfh54ay/app.bsky.graph.starterpack/3l6thoodq6j2y,at://did:plc:zxkhlfhfauvsbjtpfbfh54ay/app.bsky.graph.starterpack/3l6thq5oh6r2y,at://did:plc:zy33hvaxxt5k6vj6gqzqlpvd/app.bsky.graph.starterpack/3l6yllbjssr2e,at://did:plc:zy4g6jisvuiddiv3sxp2mc5d/app.bsky.graph.starterpack/3lb56warofu2e,at://did:plc:zy5f65xpf7xxsnmxqsljnnbs/app.bsky.graph.starterpack/3kzbgqgzsrc2f
SP1,,,,,,,,,,,,,,,,,,,,,
at://did:plc:22imtl2cambcra64j4k5bd22/app.bsky.graph.starterpack/3lbi7rkeagp27,1.0,0.935484,0.961538,1.0,1.000000,1.000000,1.000000,1.000000,1.0,1.0,...,1.000000,1.0,1.0,0.968750,1.000000,1.000000,1.000000,1.000000,1.000000,1.0
at://did:plc:237pyt6efy6mvz4rdfsdnmht/app.bsky.graph.starterpack/3lgyjx4gk362z,NaN,1.000000,0.990826,1.0,1.000000,1.000000,1.000000,1.000000,1.0,1.0,...,1.000000,1.0,1.0,0.989362,1.000000,1.000000,1.000000,1.000000,1.000000,1.0
at://did:plc:237v5xewomz3ckenyp6chhhg/app.bsky.graph.starterpack/3lb7awyj3my26,NaN,NaN,0.894231,1.0,1.000000,0.984848,0.952381,0.991525,1.0,1.0,...,0.946237,1.0,1.0,0.947368,0.989691,0.989691,0.989691,0.982143,0.964912,1.0
at://did:plc:23ngj6gipcnbf4meolvfjoto/app.bsky.graph.starterpack/3lbst2hxkw725,NaN,NaN,NaN,1.0,1.000000,1.000000,0.987500,0.992481,1.0,1.0,...,0.981982,1.0,1.0,0.973214,1.000000,1.000000,1.000000,1.000000,1.000000,1.0
at://did:plc:23sz5iq4zvn2ezbjt5ioeiej/app.bsky.graph.starterpack/3l73wh4ptap2s,NaN,NaN,NaN,NaN,0.971429,1.000000,1.000000,1.000000,1.0,1.0,...,1.000000,1.0,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
at://did:plc:zxkhlfhfauvsbjtpfbfh54ay/app.bsky.graph.starterpack/3l6thnofoce2j,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.981481,0.982143,1.0
at://did:plc:zxkhlfhfauvsbjtpfbfh54ay/app.bsky.graph.starterpack/3l6thoodq6j2y,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.981481,0.982143,1.0
at://did:plc:zxkhlfhfauvsbjtpfbfh54ay/app.bsky.graph.starterpack/3l6thq5oh6r2y,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.981481,0.982143,1.0


In [18]:
# address the NaN
distance_matrix = distance_matrix.combine_first(distance_matrix.T)

In [20]:
# Fill diagonal
import numpy as np
distance_matrix = distance_matrix.combine_first(distance_matrix.T)

distance_matrix = distance_matrix.reindex(
    index=distance_matrix.columns,
    columns=distance_matrix.columns
).copy()

for i in range(len(distance_matrix)):
    distance_matrix.iat[i, i] = 0

In [ ]:
# Convert to numpy
d = distance_matrix.values

# Choose number of clusters
k = 3

kmedoids = KMedoids(
    n_clusters=k,
    metric="precomputed",
    init="k-medoids++",
    random_state=42
)

kmedoids.fit(d)

labels = kmedoids.labels_
medoid_indices = kmedoids.medoid_indices_

In [ ]:
# Attach cluster labels to starter packs
cluster_results = pd.DataFrame({
    "starter_pack": distance_matrix.index,
    "cluster": labels
})

for k in range(2, 8):
    model = KMedoids(n_clusters=k, metric="precomputed", random_state=42)
    model.fit(D)
    score = silhouette_score(D, model.labels_, metric="precomputed")
    print(k, score)

In [ ]:
cluster_results

## Data Visualization
Source: https://plotly.com/python/network-graphs/ & ChatGPT

Because starter packs are linked through shared accounts, we model the data as a similarity network. We use modularity-based community detection, which identifies densely connected groups directly from the graph structure without requiring a predefined number of clusters. This method is more appropriate than distance-based clustering for analyzing relational recommendation overlap.

#### Network Definition

Nodes: Each node represents a single Bluesky starter pack.

Edges: An edge is created between two nodes if the corresponding starter packs share accounts. Edges represent overlap in recommended accounts between packs.
Edge Weight: Edge weights are defined using Jaccard similarity, calculated as the size of the intersection of accounts divided by the size of the union of accounts in the two packs.

**Higher weights indicate greater overlap in recommended accounts.**

To focus on meaningful structural relationships, we retain only edges above a predefined similarity threshold. This prevents weak or incidental overlaps from influencing the network structure.

In [24]:
# 1) Choose a similarity threshold
similarity_threshold = 0.15

# 2) Build graph directly from tmp
G = nx.Graph()

# add nodes (all packs that appear in tmp)
packs = pd.unique(tmp[["SP1", "SP2"]].values.ravel())
G.add_nodes_from(packs)

# add edges above threshold
edges_df = tmp[tmp["Jaccard"] >= similarity_threshold]
for _, row in edges_df.iterrows():
    G.add_edge(row["SP1"], row["SP2"], weight=row["Jaccard"])

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

# 3) Community detection (structural clusters)
communities = nx_comm.greedy_modularity_communities(G, weight="weight")
modularity = nx_comm.modularity(G, communities, weight="weight")

print("Num communities:", len(communities))
print("Modularity:", modularity)

# 4) Node -> community label
node_to_comm = {}
for i, comm in enumerate(communities):
    for node in comm:
        node_to_comm[node] = i

cluster_results = pd.DataFrame({
    "starter_pack": list(G.nodes()),
    "community": [node_to_comm.get(n, -1) for n in G.nodes()]
})

cluster_results.head()


Nodes: 4970
Edges: 35364
Num communities: 3116
Modularity: 0.4635151973759483


,starter_pack,community
0,at://did:plc:22imtl2cambcra64j4k5bd22/app.bsky...,0
1,at://did:plc:237pyt6efy6mvz4rdfsdnmht/app.bsky...,260
2,at://did:plc:237v5xewomz3ckenyp6chhhg/app.bsky...,1
3,at://did:plc:23ngj6gipcnbf4meolvfjoto/app.bsky...,1
4,at://did:plc:23sz5iq4zvn2ezbjt5ioeiej/app.bsky...,261


The similarity network exhibits strong community structure (modularity = 0.463), indicating that starter packs form densely connected structural clusters based on shared account recommendations. The large number of detected communities suggests that recommendation overlap is localized rather than globally interconnected, consistent with the presence of segmented recommendation clusters.

### Label Cleaning

In [49]:
def short_id(x, n=8):
    x = str(x)
    if "did:plc:" in x:
        x = x.split("did:plc:")[-1]
    if x.startswith("at://"):
        x = x.replace("at://", "")
    return x[:n]

# Use shortened DID for labels
label_map = {sp: short_id(sp) for sp in G.nodes()}

In [51]:
!pip3 install plotly nbformat ipykernel

import plotly.graph_objects as go
import plotly.io as pio
import networkx as nx

pio.renderers.default = "browser"

# Plot only largest connected component
largest_cc = max(nx.connected_components(G), key=len)
H = G.subgraph(largest_cc).copy()
print("Plotting nodes:", H.number_of_nodes(), "edges:", H.number_of_edges())

# Labels for plotted nodes only
label_map = {sp: short_id(sp) for sp in H.nodes()}

# Faster layout
pos = nx.spring_layout(H, seed=42, k=0.3, iterations=50)

# edges
edge_x, edge_y = [], []
for u, v in H.edges():
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]

edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    mode="lines",
    hoverinfo="none",
    line=dict(width=0.5, color="#888")
)

# nodes
node_x, node_y, node_color, node_text = [], [], [], []
for node in H.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)

    comm = node_to_comm.get(node, -1)
    node_color.append(comm)

    clean_label = label_map[node]
    node_text.append(f"<b>{clean_label}</b><br>Community: {comm}")

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode="markers",
    hoverinfo="text",
    text=node_text,
    marker=dict(
        size=8,
        color=node_color,
        colorscale="Viridis",
        showscale=True,
        colorbar=dict(title="Community")
    )
)

fig = go.Figure(
    data=[edge_trace, node_trace],
    layout=go.Layout(
        title="Starter Pack Similarity Network (Largest Connected Component)",
        showlegend=False,
        hovermode="closest",
        margin=dict(b=20, l=5, r=5, t=40),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
    )
)

fig.show()



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Plotting nodes: 1452 edges: 34070


### Analysis

In [45]:
print("Total nodes:", G.number_of_nodes())
print("Total edges:", G.number_of_edges())
print("Connected components:", nx.number_connected_components(G))
largest_cc = max(nx.connected_components(G), key=len)
G_lcc = G.subgraph(largest_cc).copy()

print("Largets Conected Component Size:", G_lcc.number_of_nodes())

Total nodes: 4970
Total edges: 35364
Connected components: 3077
Largets Conected Component Size: 1452


### What these numbers mean:

The similarity network contains 4,970 starter packs connected by 35,364 similarity edges. The graph is highly fragmented, consisting of 3,077 connected components. The largest connected component (LCC) contains 1,452 nodes, representing approximately 29% of all starter packs. This indicates a core–periphery structure: while a substantial central ecosystem of overlapping packs exists, the majority of starter packs form small, disconnected clusters. This suggests both **niche fragmentation and a central zone of shared exposure**, complicating a simple echo chamber narrative.


In [26]:
community_sizes = sorted([len(c) for c in communities], reverse=True)

print("Largest 10 community sizes:", community_sizes[:10])
print("Median community size:", sorted(community_sizes)[len(community_sizes)//2])

Largest 10 community sizes: [784, 443, 80, 49, 48, 26, 19, 17, 13, 13]
Median community size: 1


Community detection reveals a highly fragmented yet strongly modular network (modularity = 0.463). While the median community size is 1, indicating that many starter packs are structurally isolated, a small number of large communities (sizes 784 and 443) account for a significant portion of interconnected packs. This distribution reflects a heterogeneous structure: most packs are niche and weakly connected, but several dense clusters form around recurring account overlaps. Such clustering patterns are consistent with localized recommendation bubbles embedded within a broader, fragmented ecosystem.

### Justification for Analyzing the Largest Connected Component

Initially, we explored distance-based clustering using K-Medoids on the pairwise Jaccard distance matrix. However, silhouette scores were near zero, indicating weak separation in the distance space. This suggests that starter packs do not form clearly separable “blob-like” clusters in Euclidean distance space.

Because our data is naturally relational—defined by shared account overlap, it is more appropriately modeled as a similarity network rather than a geometric clustering problem. We therefore choose to adopt modularity-based community detection, which operates directly on the graph structure and identifies densely connected groups without requiring a predefined number of clusters.

Furthermore, the full network contains many isolated or weakly connected starter packs. To improve interpretability and focus on substantive structural relationships, we restrict visualization to the largest connected component. This component captures the main interconnected core of the recommendation ecosystem while excluding isolated packs that do not meaningfully contribute to broader clustering patterns.